# Lab 17 — Robustness Mitigation Comparison

*AI Cybersecurity Masterclass · Lab 17 · AI60secLab*

**Research question:** Which defensive treatment best preserves detection under controlled input stress without sacrificing too much clean performance?

We compare:
1. baseline;
2. validation-selected threshold;
3. bounded noisy training augmentation.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.evaluation import binary_metrics
from src.thresholds import threshold_sweep, choose_threshold, apply_threshold
from src.adversarial import (
    available_numeric_attack_features,
    training_bounds,
    bounded_random_perturbation,
    augment_attacks_with_noise,
)

train, test = load_unsw(DATA_DIR)

X_full, y_full = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_full, X_test = align_columns(X_full, X_test)

X_full = drop_identifier_like_columns(X_full)
X_test = drop_identifier_like_columns(X_test)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    stratify=y_full,
    random_state=42,
)

In [ ]:
baseline = random_forest_pipeline(X_train)
baseline.fit(X_train, y_train)

valid_scores = baseline.predict_proba(X_valid)[:, 1]
sweep = threshold_sweep(
    y_valid,
    valid_scores,
    beta=2.0,
)
selected_threshold, _ = choose_threshold(
    sweep,
    objective="f_beta",
)

print(
    "Validation-selected threshold:",
    round(selected_threshold, 3),
)

In [ ]:
perturbation_columns = available_numeric_attack_features(X_train)
bounds = training_bounds(X_train, perturbation_columns)

X_augmented, y_augmented = augment_attacks_with_noise(
    X_train,
    y_train,
    columns=perturbation_columns,
    epsilon=0.05,
    copies=1,
    bounds=bounds,
    random_state=42,
)

augmented_model = random_forest_pipeline(X_augmented)
augmented_model.fit(X_augmented, y_augmented)

In [ ]:
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

X_stressed = bounded_random_perturbation(
    X_test,
    columns=perturbation_columns,
    epsilon=0.10,
    bounds=bounds,
    random_state=99,
)

rows = []

for condition, X_eval in [
    ("clean", X_test),
    ("stressed", X_stressed),
]:
    baseline_scores = baseline.predict_proba(X_eval)[:, 1]
    baseline_default = apply_threshold(
        baseline_scores, 0.50
    )
    baseline_tuned = apply_threshold(
        baseline_scores,
        selected_threshold,
    )

    augmented_scores = augmented_model.predict_proba(X_eval)[:, 1]
    augmented_pred = apply_threshold(
        augmented_scores, 0.50
    )

    for strategy, pred, score in [
        ("Baseline", baseline_default, baseline_scores),
        ("Threshold tuned", baseline_tuned, baseline_scores),
        ("Noise augmented", augmented_pred, augmented_scores),
    ]:
        metrics = binary_metrics(
            y_test,
            pred,
            score,
        )
        rows.append({
            "condition": condition,
            "strategy": strategy,
            "precision": metrics.precision,
            "recall": metrics.recall,
            "f1": metrics.f1,
            "false_negative_rate": metrics.false_negative_rate,
        })

comparison = pd.DataFrame(rows)
display(comparison.round(4))

In [ ]:
pivot = comparison.pivot(
    index="strategy",
    columns="condition",
    values="recall",
)

pivot["recall_degradation"] = (
    pivot["clean"] - pivot["stressed"]
)

display(
    pivot.sort_values("recall_degradation").round(4)
)

## Interpretation

Which strategy had the strongest clean recall? Which had the smallest degradation? Did robustness improvement create a precision cost?

**Takeaway:** robustness mitigation is a tradeoff problem and must be evaluated under both clean and stressed conditions.